## tl;dr

For POIs queried as `id + geometry`, the Parquet 1 MiB dictionary default is the transfer cliff. With 64k RowGroups and 512-row Pages, a 32 KiB dictionary cap reduced full-data query transfer by 62–70% for only 1.7% file growth. A 16 KiB cap saved another 11% on the 10% sample but cost materially more file space, making 32 KiB the practical knee. 32k RowGroups remained better than 64k for this POI workload, while 64k remains a defensible general default.

## Context & Methods

The decision is whether dictionary-page sizing can prevent a high-cardinality projected column such as `id` from dominating sparse PageIndex reads. Layout candidates hold Page rows at 512 and vary RowGroup rows (32,768 or 65,536) and dictionary cap (16, 32, 64, 128, 256, or 1,024 KiB).

### Key Assumptions

- The controlled sweep uses the same deterministic global 10% sample as the earlier POI benchmark: `hash(id) % 10 = 0`.
- Queries cover eight Japanese cities at 1,000 m, 100 m, and 10 m resolution with `id,geometry` projection.
- The 25 ms RTT model delays each underlying range read; concurrent reads overlap. Bandwidth estimates add serial payload time and are not an HTTP/2 or HTTP/3 simulator.
- The full-data comparison isolates 64k/512/32KiB against the existing 64k/512/1MiB file.

## Data

Raw reader runs, Parquet metadata, derived CSVs, and validation checks live beside this notebook. Candidate Parquet files are intentionally kept in `/tmp/cogp-pois-dictionary-bench` because they total several gigabytes.

In [1]:
from pathlib import Path
import csv
import json

bench = Path.cwd() / 'benchmarks/2026-09-07-pois-dictionary-pages'
with (bench / 'controlled-summary.csv').open() as handle:
    controlled = list(csv.DictReader(handle))
with (bench / 'full-summary.csv').open() as handle:
    full = list(csv.DictReader(handle))
validation = json.loads((bench / 'validation.json').read_text())
print(f'controlled layouts: {len(controlled)}')
print(f'full-data resolution cuts: {len(full)}')
print(f'validation checks: {len(validation["checks"])}')


controlled layouts: 12
full-data resolution cuts: 3
validation checks: 11


## Results

The table below keeps exact byte counts in the CSV and prints the decision-facing aggregate for each controlled candidate. Negative percentages mean improvement versus the 1 MiB cap at the same RowGroup size.

In [2]:
print('RG rows | dict KiB | file delta | query-byte delta | mean RTT25 ms | est. 100 Mbps ms')
for row in controlled:
    print(
        f'{int(row["rowGroupRows"]):>7} | '
        f'{int(row["dictionaryLimitKiB"]):>8} | '
        f'{float(row["fileSizeVs1MiBPct"]):>+9.1f}% | '
        f'{float(row["queryBytesVs1MiBPct"]):>+15.1f}% | '
        f'{float(row["meanRtt25Ms"]):>13.1f} | '
        f'{float(row["estimated100MbpsMs"]):>15.1f}'
    )


RG rows | dict KiB | file delta | query-byte delta | mean RTT25 ms | est. 100 Mbps ms
  32768 |       16 |      +6.3% |           -60.9% |         279.6 |           332.2
  32768 |       32 |      +3.1% |           -55.2% |         280.5 |           340.7
  32768 |       64 |      +3.1% |           -44.3% |         281.0 |           355.9
  32768 |      128 |      +3.3% |           -27.1% |         280.7 |           378.7
  32768 |      256 |      +3.5% |            +0.0% |         281.1 |           415.5
  32768 |     1024 |      +0.0% |            +0.0% |         282.5 |           417.0
  65536 |       16 |      +5.6% |           -73.9% |         300.4 |           359.8
  65536 |       32 |      +2.8% |           -70.6% |         302.0 |           368.7
  65536 |       64 |      +1.0% |           -64.3% |         302.5 |           383.6
  65536 |      128 |      +1.1% |           -53.6% |         303.9 |           409.3
  65536 |      256 |      +1.2% |           -31.5% |         307

The controlled sweep identifies 32 KiB as the knee: 16 KiB minimizes transfer, but its additional file-size cost is much larger than the modest extra transfer saving. At equal dictionary caps, 32k RowGroups transfer less than 64k and are usually as fast or faster.

In [3]:
print('Full 30M rows: resolution | bytes change | requests change | local-time change | 100 Mbps estimate')
for row in full:
    estimate_change = (
        float(row['candidateEstimated100MbpsMs']) / float(row['baselineEstimated100MbpsMs']) - 1
    ) * 100
    print(
        f'{int(row["resolutionMeters"]):>4} m | '
        f'{float(row["queryBytesChangePct"]):>+6.1f}% | '
        f'{float(row["requestsChangePct"]):>+6.1f}% | '
        f'{float(row["localMsChangePct"]):>+6.1f}% | '
        f'{estimate_change:>+6.1f}%'
    )


Full 30M rows: resolution | bytes change | requests change | local-time change | 100 Mbps estimate
1000 m |  -62.3% |   +6.1% |  -23.7% |  -23.9%
 100 m |  -67.3% |  +17.5% |  -15.9% |  -23.7%
  10 m |  -70.0% |  +22.9% |   -9.9% |  -23.8%


## Takeaways

- Set the general dictionary-page cap to 32 KiB when PageIndex-driven sparse reads are a primary workload. The cap protects any high-cardinality projected column, not only a field named `id`; low-cardinality dictionaries stay below the cap naturally.
- Keep 64k RowGroups as the general default if desired, but use a 32k POI profile. The dictionary cap fixes the catastrophic payload component; RowGroup 32k still improves POI locality and request scheduling.
- Do not use 16 KiB by default yet. It is a bandwidth-first profile whose extra file growth needs a broader attribute workload before adoption.
- A future per-column pilot could retain a larger dictionary for compression-friendly columns such as tags while capping only high-cardinality columns.

In [4]:
failed = [check for check in validation['checks'] if check['mismatches']]
assert not failed, failed
assert all(int(row['num_rows']) == 3_004_499 for row in csv.DictReader((bench / 'file-metadata.csv').open()))
print('PASS: all candidates preserve query row counts; all controlled files contain 3,004,499 rows.')


PASS: all candidates preserve query row counts; all controlled files contain 3,004,499 rows.
